# SA²-v2 on 7×7 CLIP tokens — **train on Nexar, zero-shot on DADA-2000**, seed 2024

Rule (written before any number): `core/docs/final/SA2V2_RUN.md`. Upload branch `v_final_stp_v2` to
`Thesis-Final/kat-vad`. `Thesis/`, `Thesis-V2/`, `Thesis-V5/` are read-only; everything new lands in `Thesis-Final/`.

| arm | what | checkpoint |
|---|---|---|
| **W0** (control) | CLIP + V2-S via `W_u`, no spatial module | `outputs/wu/W0/s2024`, reused; read `outputs/zs/eval/dada/W0` |
| **SV2** | W0 + SA²-v2 (proposal row R7): priors + text term + `L_patch` + head on `x^AS` + motion dropout | `outputs/sv2/SV2/s2024` |

| part | step | what | GPU | estimate |
|---|---|---|---|---|
| A | 1–3 | setup, preflight | — | 5–10 min |
| A | 4 | Nexar: pool `cgrid14` → 7×7, level constants | no | 15–30 min |
| A | 5 | DADA: one CLIP pass over 16,090 frames | **yes** | 10–20 min |
| A | 6 | train SV2 (1,749 steps). **Stop and look if the log shows > 2 s/step** | **yes** | 15–45 min |
| A | 7 | DADA read + dump | yes | 5–10 min |
| A | 8 | read-out → **paste `REPORTS/sa2v2_readout.md` to Claude and stop** | — | |
| B | 9 | C2: shuffled labels. Set `RUN_C2 = True` after reading part A | **yes** | 15–45 min |
| C | 10 | DoTA-CAP-dev: read + maps. Set `RUN_DOTA = True` after reading part A | **yes** | 10–15 min |

Estimates only; nothing has been timed. Every step skips what is already on Drive, so re-run after a disconnect.
One seed ⇒ every sentence is **PRELIMINARY (n = 1 seed)** and adopts nothing.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, KNN, v2 A2 runs
DRIVE_V5 = '/content/drive/MyDrive/Thesis-V5'     # read-only: 14 x 14 CLIP token grids (branch v5-r7)
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every cache / output this notebook writes
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_stp_v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'SV2_ARM'), 'this upload predates SA²-v2 -- re-upload kat-vad/'
assert (REPO / 'core/docs/final/SA2V2_RUN.md').is_file(), 'no SA2V2_RUN.md (the read-out rule) in this upload'

SEED, ARM, SHUF = constants.SV2_SEED, constants.SV2_ARM, constants.SV2_SHUFFLED_ARM
ENC      = constants.V5_ENCODER
C        = constants.NEXAR_CONSTRUCTION_WINDOW
DATASET  = constants.NEXAR_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
S_N, S_DADA, S_DOTA = constants.STP_NEXAR_STRIDE, constants.ZS_STRIDE, constants.STP_DOTA_STRIDE

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
A2_RUNS  = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'A2'
GRIDS    = Path(DRIVE_V5) / 'cache' / 'video_grid' / 'clip_vit_b16'
GRID_N   = GRIDS / f'{DATASET}_s{S_N}_cgrid14'
GRID_D   = GRIDS / f'{constants.DOTA_CAP_DATASET}_s{S_DOTA}_cgrid14'
T2       = constants.DATA_ROOT / T2NAME
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
ARC_DADA = Path(DRIVE_F) / 'data' / constants.ZS_FRAMES_DIRNAME / T2NAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
ZS_ROOT  = constants.OUTPUT_ROOT / constants.ZS_OUTPUT_DIRNAME
W0_RUN   = constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / constants.ZS_CONTROL_ARM / f's{SEED}'
W0_DADA  = ZS_ROOT / 'eval' / 'dada' / constants.ZS_CONTROL_ARM
R4_DADA  = ZS_ROOT / 'eval' / 'dada' / constants.ZS_SA2_ARM
W0_DOTA  = constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / 'eval' / 'dota' / CAP_DEV / constants.ZS_CONTROL_ARM

from core.data import stp_patches as stp_data      # noqa: E402

CELL_N, CELL_DADA = stp_data.cell_cache_dir(DATASET, S_N), stp_data.cell_cache_dir(T2NAME, S_DADA)
CELL_DOTA = stp_data.cell_cache_dir(constants.DOTA_CAP_DATASET, S_DOTA)
ROOT     = constants.OUTPUT_ROOT / constants.SV2_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
RUN_DIR  = {ARM: ROOT / ARM / f's{SEED}', SHUF: ROOT / SHUF / f's{SEED}'}
LOCAL    = Path('/content/sv2')
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']     # 1,749 = 33 x 53
for name, path in (('Nexar_window', DATA), ('Nexar A2 bake', A2_NEXAR), ('DADA A2 bake', A2_DADA), ('KNN', KNN),
                   ('v2 A2 runs', A2_RUNS), ('Nexar tokens', GRID_N), ('T2 meta', T2 / constants.META_FILENAME),
                   ('T2 CLIP', T2_CLIP), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
                   ('DADA frames', ARC_DADA / constants.ZS_FRAMES_MANIFEST), ('W0 run', W0_RUN / 'metrics.jsonl'),
                   ('W0 DADA read', W0_DADA / 'clip_aucs.json')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:16s} {path}')
print(f'steps {STEPS} ({EPOCHS} epochs) | cells -> {CELL_N} | {CELL_DADA}\nruns/reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 5-7, 9, 10 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import threading
import time

import numpy as np
import torch
import yaml

from core.tools.dada_whole_eval import test_sources    # noqa: E402
from core.tools.sa2v2_cells import training_sources    # noqa: E402
from core.tools.v5_stage1 import _flat                 # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF (absl eats INFO logs)


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def rows(d, v):
    return int(np.load(d / f'{v}.npy', mmap_mode='r').shape[0])


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def is_verified(drive_dir):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == STEPS and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


class GpuPeak:
    """Peak GPU memory of a block, sampled with nvidia-smi (proposal analysis 6: cost)."""

    def __enter__(self):
        self.peak, self.stop = 0, threading.Event()
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()
        return self

    def _loop(self):
        while not self.stop.wait(5):
            out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                 capture_output=True, text=True).stdout.split()
            if out:
                self.peak = max(self.peak, int(out[0]))

    def __exit__(self, *exc):
        self.stop.set()
        self.thread.join()
        print(f'peak GPU memory: {self.peak} MiB')


# ids and the preflight: nothing below runs on a wrong corpus
NEXAR_IDS = training_sources(DATA)
DADA_IDS = test_sources(T2)
assert len(NEXAR_IDS) == 451, f'{len(NEXAR_IDS)} Nexar training sources, expected 451'
assert len(DADA_IDS) == 370, len(DADA_IDS)
miss = [v for v in NEXAR_IDS if not (GRID_N / f'{v}.npy').exists()]
assert not miss, f'{len(miss)} Nexar sources have no token grid in {GRID_N}: {miss[:5]}'
miss = [v for v in DADA_IDS if not (A2_DADA / f'{v}.npy').exists()]
assert not miss, f'{len(miss)} DADA sources have no A2 bake: {miss[:5]}'
g = np.load(GRID_N / f'{NEXAR_IDS[0]}.npy', mmap_mode='r')
assert g.size == g.shape[0] * 14 * 14 * 512, f'token grid has shape {g.shape}, expected (L, 14, 14, 512)'
man = json.loads((ARC_DADA / constants.ZS_FRAMES_MANIFEST).read_text())
assert man['pass'] and man['stride'] == S_DADA, man
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA)):
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert (m['crn'], m['motion']) == (constants.V2_OFF, ENC), f'{name} bake is {m}'
same = {json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())['train_ids_sha1'] for d in (A2_NEXAR, A2_DADA)}
assert len(same) == 1, f'the bakes carry different statistics: {same}'
ID_N, ID_DADA = write_ids(LOCAL / 'nexar.txt', NEXAR_IDS), write_ids(LOCAL / 'dada.txt', DADA_IDS)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in DATA.glob('*.json'):               # defs.json etc.: the definitions the arms trained with
    shutil.copy2(f, L_DEFS / f.name)
print(f'preflight OK: {len(NEXAR_IDS)} Nexar sources with tokens, {len(DADA_IDS)} DADA sources, shared statistics')

## 4. Nexar cells: 14×14 tokens → 7×7 (CPU), level constants

`sa2v2_cells pool` averages each 2×2 block of the `v5-r7` token grid and checks every row count against the A2 bake
(C13). `stats` freezes `m, s` of Mo and ρ over the 451 training sources; the checkpoint carries them to every test set.

In [ ]:
t0 = time.time()
run([sys.executable, '-m', 'core.tools.sa2v2_cells', 'pool', '--grid-dir', GRID_N, '--ids-file', ID_N,
     '--input-dir', A2_NEXAR, '--dataset', DATASET, '--stride', S_N, '--out-dir', CELL_N], log=LOCAL / 'pool_nexar.log')
if not (CELL_N / constants.SV2_LEVEL_FILENAME).exists():
    run([sys.executable, '-m', 'core.tools.sa2v2_cells', 'stats', '--cell-dir', CELL_N, '--data-dir', DATA])
print('level constants:', json.loads((CELL_N / constants.SV2_LEVEL_FILENAME).read_text()))
print(f'Nexar cells in {(time.time() - t0) / 60:.1f} min')

## 5. DADA cells (GPU): one CLIP pass per frame

Frames from `ZS_frames/DADA2000_orig`. A video is stored only if the projected CLS equals `image_embeds` (G1) and
reproduces the video's own CLIP rows (row count, every row cos ≥ 0.95, mean ≥ 0.99). **Re-run after a disconnect.**

In [ ]:
assert torch.cuda.is_available(), 'torch sees no GPU'
t0 = time.time()
todo = [v for v in DADA_IDS if not (CELL_DADA / f'{v}.npy').exists()]
print(f'DADA cells: {len(DADA_IDS) - len(todo)}/{len(DADA_IDS)} done')
if todo:
    arc = stage(ARC_DADA, LOCAL / 'arc_dada')
    clip = stage(T2_CLIP, LOCAL / 'clip_dada', DADA_IDS)
    run([sys.executable, '-m', 'core.tools.sa2v2_cells', 'extract', '--archive-dir', arc, '--work-dir', LOCAL / 'unzip',
         '--clip-dir', clip, '--ids-file', ID_DADA, '--report', REPORTS / 'sv2_extract_dada.json', '--out-dir', CELL_DADA],
        log=LOCAL / 'extract_dada.log')
    flush_drive()
bad = [v for v in DADA_IDS if rows(CELL_DADA, v) != rows(A2_DADA, v)]
assert not bad, f'DADA cells != A2 bake rows on {bad[:5]}'
print(f'DADA cells OK, rows == bake rows, {(time.time() - t0) / 60:.1f} min')

## 6. Train SV2 (GPU)

v2 Nexar A2's recipe + `stp.sa2v2=true`, prefix off. **Watch the log: past 2 s/step, stop and paste it to Claude.**
Then J10, the config diff against v2 A2 s2024, and the two frozen buffers.

In [ ]:
def train_cmd(out_dir, data, knn):
    cmd = [sys.executable, '-m', 'core.train',
           '--set', 'train.stage=2', '--set', f'train.seed={SEED}',
           '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
           '--set', f'data.dataset={DATASET}',
           '--set', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
           '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
           '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
           '--set', 'data.is_egocentric=false',
           '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}',
           '--set', 'stp.sa2v2=true',
           '--data-dir', data, '--clip-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar', NEXAR_IDS),
           '--patch-dir', stage(CELL_N, LOCAL / 'stage' / 'cell_nexar', NEXAR_IDS), '--output-dir', out_dir]
    return cmd + (['--knn-cache', knn] if knn else [])


def train_arm(arm, data, knn):
    drive_dir, local = RUN_DIR[arm], LOCAL / 'runs' / arm
    if is_verified(drive_dir):
        print(f'{arm} s{SEED}: verified on Drive -- not retrained')
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    with GpuPeak():
        run(train_cmd(local, data, knn) + resume, log=LOCAL / f'train_{arm}.log')
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{arm}: J10 locally'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, drive_dir / name)
    flush_drive()
    assert ckpt_step(drive_dir / 'checkpoint_last.pt') == STEPS, f'{arm}: Drive copy is not step {STEPS}'
    (LOCAL / 'verified.json').write_text(json.dumps(
        {'step': STEPS, 'ckpt_bytes': (drive_dir / 'checkpoint_last.pt').stat().st_size}))
    publish(LOCAL / 'verified.json', drive_dir / 'verified.json')
    print(f'{arm}: {(time.time() - t0) / 60:.0f} min | entropy {last.get("sv2_entropy")} | gamma {last.get("sv2_gamma")} '
          f'| beta1 {last.get("sv2_beta1")} | x^AS / c {last.get("sv2_xas_ratio")}')


def check_run(arm):
    d = RUN_DIR[arm]
    payload = torch.load(d / 'checkpoint_last.pt', map_location='cpu', weights_only=False)
    assert payload['global_step'] == metrics_last(d / 'metrics.jsonl')['global_step'] == STEPS, f'{arm}: J10'
    state = payload['model']
    level = json.loads((CELL_N / constants.SV2_LEVEL_FILENAME).read_text())
    want = [level['mo_mean'], level['mo_std'], level['rho_mean'], level['rho_std']]
    assert np.allclose(state['sa2v2.level'].numpy(), want, rtol=1e-5), f'{arm}: level constants {state["sa2v2.level"]}'
    assert not torch.equal(state['sa2v2.text_projection'], torch.eye(512)), f'{arm}: tau_delta has no CLIP projection'
    ref = _flat(yaml.safe_load((A2_RUNS / f's{SEED}' / 'config.yaml').read_text()))
    got = _flat(yaml.safe_load((d / 'config.yaml').read_text()))
    diff = {k: (ref.get(k), got.get(k)) for k in set(ref) | set(got)
            if not k.startswith(('stp.', 'v5.')) and k not in ('model.motion_init_std', 'train.motion_lr_mult')
            and ref.get(k) != got.get(k) and not (k == 'model.motion_dropout' and not got.get(k))}
    assert not diff, f'{arm} differs from v2 A2 s{SEED} beyond stp / v5 / W_u knobs: {diff}'
    assert got['stp.sa2v2'] is True and got['stp.sa2'] is False, arm
    assert got['stp.text_prompt'] == constants.STP_PROMPT_RELEASED, f'{arm}: prefix is not OFF'
    print(f'{arm}: J10 {STEPS}, level + projection stored, config == v2 A2 s{SEED} + stp.sa2v2, prefix OFF')


data_local = stage(DATA, LOCAL / 'stage' / 'data')
knn_local = stage(KNN, LOCAL / 'stage' / 'knn.npz')
train_arm(ARM, data_local, knn_local)
check_run(ARM)

## 7. DADA-2000: frame read + dump (GPU)

Same protocol and definitions as W0's stored read. The dump holds the map, the head on `x^AS` and on the cell mean,
top-3 q and entropy; the read-out checks that its scores equal the frame read's.

In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
DADA_OUT, CKPT = ROOT / 'eval' / 'dada', RUN_DIR[ARM] / 'checkpoint_last.pt'
if not (DADA_OUT / ARM / 'dada_readout.json').exists() or not (DADA_OUT / f'maps_{ARM}.npz').exists():
    assert torch.cuda.is_available(), 'torch sees no GPU'
    bake, cells = stage(A2_DADA, LOCAL / 'bake_dada', DADA_IDS), stage(CELL_DADA, LOCAL / 'cell_dada', DADA_IDS)
    local = LOCAL / 'dada' / ARM
    run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', CKPT, '--t2-dir', L_T2,
         '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', bake, '--patch-dir', cells,
         '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / 'dada.log')
    publish(local, DADA_OUT / ARM)
    dump = LOCAL / 'dada_dump'
    run([sys.executable, '-m', 'core.tools.sa2v2_eval', '--name', ARM, '--ckpt', CKPT, '--input-dir', bake,
         '--cell-dir', cells, '--data-dir', L_DEFS, '--ids-file', ID_DADA, '--stride', S_DADA, '--out-dir', dump],
        log=LOCAL / 'dada_dump.log')
    publish(dump, DADA_OUT)
    flush_drive()
r = json.loads((DADA_OUT / ARM / 'dada_readout.json').read_text())
print(f"dada {ARM}: macro {r['macro']['mean']:.4f} over {r['two_class']} sources | micro {r['micro']:.4f}")

## 8. Read-out → **paste `REPORTS/sa2v2_readout.md` to Claude, then stop**

CPU only. Parts B and C below are off until the main read has been looked at. Re-run this cell after each of them.

In [ ]:
RUN_C2, RUN_DOTA = False, False     # set after reading part A


def readout():
    extra = []
    if (RUN_DIR[SHUF] / 'metrics.jsonl').exists():
        extra += ['--shuffled-run-dir', RUN_DIR[SHUF]]
    if (R4_DADA / 'clip_aucs.json').exists():
        extra += ['--r4-dada', R4_DADA]
    if (ROOT / 'eval' / 'dota' / ARM / 'clip_aucs.json').exists():
        extra += ['--w0-dota', W0_DOTA, '--sv2-dota', ROOT / 'eval' / 'dota' / ARM,
                  '--maps-md', ROOT / 'eval' / 'dota' / 'maps' / 'maps_readout.md']
    run([sys.executable, '-m', 'core.tools.sa2v2_readout', '--w0-dada', W0_DADA, '--sv2-dada', DADA_OUT / ARM,
         '--dump', DADA_OUT / f'maps_{ARM}.npz', '--run-dir', RUN_DIR[ARM], '--out-dir', REPORTS, *extra])
    print((REPORTS / 'sa2v2_readout.md').read_text())


readout()
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
drive.mount('/content/drive')
print('Drive flushed -- paste REPORTS/sa2v2_readout.md to Claude')

## 9. Part B: control C2, shuffled labels (GPU)

Same recipe; labels and class names permuted across bags (seed fixed in `constants`). No KNN cache: its neighbours were
built from the real labels. A movement in the real run counts only where it exceeds this run's.

In [ ]:
if RUN_C2:
    shuf_data = LOCAL / 'stage' / 'data_shuffled'
    if not (shuf_data / 'shuffle_report.json').exists():
        run([sys.executable, '-m', 'core.tools.sa2v2_cells', 'shuffle-labels', '--data-dir', DATA, '--out-dir', shuf_data])
    print(json.loads((shuf_data / 'shuffle_report.json').read_text()))
    train_arm(SHUF, shuf_data, None)
    check_run(SHUF)
    readout()
    flush_drive()
else:
    print('RUN_C2 is False')

## 10. Part C: DoTA-CAP-dev, same checkpoint (GPU)

Protocol B per clip against W0's stored read, then STAUC and the pointing game of `q` beside Mo, uniform and centre,
with `v5_maps read`, the code that read the earlier token test.

In [ ]:
from core.data.v2_splits import load_split    # noqa: E402

if RUN_DOTA:
    assert torch.cuda.is_available(), 'torch sees no GPU'
    for name, path in (('DoTA tokens', GRID_D), ('A2 bake', A2_DOTA), ('DoTA s1', DOTA_S1), ('labels', DOTA / 'labels_s8'),
                       ('W0 read', W0_DOTA / 'clip_aucs.json')):
        assert path.exists(), f'missing {name}: {path}'
    CAP_IDS = sorted(load_split(CAP_DEV))
    ID_DOTA = write_ids(LOCAL / 'dota.txt', CAP_IDS)
    run([sys.executable, '-m', 'core.tools.sa2v2_cells', 'pool', '--grid-dir', GRID_D, '--ids-file', ID_DOTA,
         '--input-dir', A2_DOTA, '--dataset', constants.DOTA_CAP_DATASET, '--stride', S_DOTA, '--out-dir', CELL_DOTA],
        log=LOCAL / 'pool_dota.log')
    bake, cells = stage(A2_DOTA, LOCAL / 'bake_dota', CAP_IDS), stage(CELL_DOTA, LOCAL / 'cell_dota', CAP_IDS)
    s1, labels = stage(DOTA_S1, LOCAL / 'dota_s1', CAP_IDS), stage(DOTA / 'labels_s8', LOCAL / 'dota_labels')
    META, SPLIT = DOTA / 'metadata_val.json', DOTA / 'val_split.txt'
    OUT = ROOT / 'eval' / 'dota'
    if not (OUT / ARM / 'clip_aucs.json').exists():
        local = LOCAL / 'dota' / ARM
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', CKPT, '--input-dir', bake,
             '--patch-dir', cells, '--s1-dir', s1, '--metadata', META, '--split-file', SPLIT, '--data-dir', labels,
             '--split', CAP_DEV, '--out-dir', local, '--device', 'auto'], log=LOCAL / 'dota.log')
        publish(local, OUT / ARM)
    dump = LOCAL / 'dota_dump'
    if not (OUT / 'maps' / 'maps_readout.md').exists():
        run([sys.executable, '-m', 'core.tools.sa2v2_eval', '--name', ARM, '--ckpt', CKPT, '--input-dir', bake,
             '--cell-dir', cells, '--data-dir', labels, '--ids-file', ID_DOTA, '--stride', S_DOTA, '--s1-dir', s1,
             '--baseline-maps', '--out-dir', dump], log=LOCAL / 'dota_dump.log')
        ANN = DOTA / 'raw_annotations' / 'annotations'
        if not ANN.is_dir():
            ANN = LOCAL / 'dota_meta' / 'annotations'
            ANN.parent.mkdir(parents=True, exist_ok=True)
            subprocess.run(f'cd {ANN.parent} && curl -sSL -o ann.zip https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip && unzip -q ann.zip && rm ann.zip',
                           shell=True, check=True)
        print('annotations:', len(list(ANN.glob('*.json'))))
        maps = [x for n in (ARM, 'Mo', 'uniform', 'centre') for x in ('--run', n, SEED, dump / f'maps_{n}.npz')]
        run([sys.executable, '-m', 'core.tools.v5_maps', 'read', *maps, '--stride', S_DOTA, '--metadata', META,
             '--split-file', SPLIT, '--annotations-dir', ANN, '--out-dir', LOCAL / 'maps'])
        publish(LOCAL / 'maps', OUT / 'maps')
        publish(dump / f'maps_{ARM}.npz', OUT / 'maps' / f'maps_{ARM}.npz')
    readout()
    flush_drive()
else:
    print('RUN_DOTA is False')